# Hands-On Lab: Binary & Multi-class Classification

**Week 6 Capstone Lab** | Difficulty: ⭐⭐⭐⭐ | Estimated time: 4 hours

---

## Learning Objectives
By the end of this lab you will be able to:
1. Implement logistic regression **from scratch** using gradient descent with L2 regularization
2. Visualize and interpret **decision boundaries** for five different classifiers
3. Understand why different classifiers succeed or fail on different dataset geometries
4. Apply logistic regression to **MNIST digit classification** (3 vs 8)
5. Tune the **classification threshold** to optimise precision, recall, or F1/F2
6. Verify gradient correctness with **numerical finite differences**

---

## Lab Outline
| Section | Topic | Cells |
|---------|-------|-------|
| 0 | Setup & Imports | 2 |
| 1 | Logistic Regression from Scratch | 6 |
| 2 | Decision Boundary Visualization Gallery | 7 |
| 3 | Numerical Gradient Verification | 2 |
| 4 | Multi-class Classification | 4 |
| 5 | MNIST Binary Classification (3 vs 8) | 4 |
| 6 | Threshold Tuning | 4 |
| 7 | Classifier Comparison Summary | 3 |
| 8 | Self-Check Questions | 1 |

In [ ]:
# ── Core numerical & data libraries ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── sklearn: datasets ────────────────────────────────────────────────────────
from sklearn.datasets import (
    make_blobs, make_moons, make_circles, fetch_openml
)

# ── sklearn: preprocessing & model selection ─────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ── sklearn: classifiers ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

# ── sklearn: metrics ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, auc
)

# ── Global settings ───────────────────────────────────────────────────────────
np.random.seed(42)                     # reproducibility throughout
sns.set_theme(style='whitegrid')       # consistent plot style
plt.rcParams['figure.dpi'] = 110       # crisper inline figures

print('All imports successful.')

---
## Section 1: Logistic Regression from Scratch

We implement gradient descent for binary cross-entropy loss with optional L2 regularisation (weight decay).  
The update rule for parameters $\boldsymbol{\theta}$ is:

$$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \alpha \left[\frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{y}} - \mathbf{y}) + \lambda\boldsymbol{\theta}\right]$$

Note: the bias term $\theta_0$ is **not** regularised.

In [ ]:
class LogisticRegressionScratch:
    """Binary logistic regression via gradient descent with optional L2 regularisation."""

    def __init__(self, lr=0.1, n_iter=1000, lambda_=0.0):
        self.lr = lr           # learning rate α
        self.n_iter = n_iter   # number of gradient-descent steps
        self.lambda_ = lambda_ # L2 regularisation strength (0 = no regularisation)

    def _sigmoid(self, z):
        """Numerically stable sigmoid: clips z to avoid overflow in exp."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        """Train on feature matrix X (n×p) and binary label vector y (n,)."""
        n, p = X.shape
        X_b = np.column_stack([np.ones(n), X])   # prepend bias column → shape (n, p+1)
        self.theta_ = np.zeros(X_b.shape[1])      # initialise all params to zero
        self.loss_history_ = []                    # track loss each iteration

        for i in range(self.n_iter):
            z = X_b @ self.theta_                  # linear combination (n,)
            y_hat = self._sigmoid(z)               # predicted probabilities

            # Gradient of cross-entropy w.r.t. θ (elegant: same form as linear regression)
            gradient = (1 / n) * X_b.T @ (y_hat - y)   # shape (p+1,)

            # L2 regularisation penalty — bias θ[0] is intentionally excluded
            reg_term = self.lambda_ * self.theta_.copy()
            reg_term[0] = 0                        # do NOT penalise the bias

            self.theta_ -= self.lr * (gradient + reg_term)  # parameter update

            # Recompute y_hat after update for accurate loss logging
            y_hat_log = self._sigmoid(X_b @ self.theta_)
            eps = 1e-15                            # avoid log(0)
            y_hat_log = np.clip(y_hat_log, eps, 1 - eps)
            loss = -np.mean(y * np.log(y_hat_log) + (1 - y) * np.log(1 - y_hat_log))
            self.loss_history_.append(loss)

        return self

    def predict_proba(self, X):
        """Return predicted probability P(y=1|X) for each sample."""
        X_b = np.column_stack([np.ones(len(X)), X])
        return self._sigmoid(X_b @ self.theta_)

    def predict(self, X, threshold=0.5):
        """Return hard binary predictions using the given decision threshold."""
        return (self.predict_proba(X) >= threshold).astype(int)

print('LogisticRegressionScratch class defined.')

In [ ]:
# ── Synthetic spam dataset ────────────────────────────────────────────────────
# 4 features: word_freq_free, char_freq_exclamation, capital_run_avg, link_count
np.random.seed(42)
n_samples = 500

# Spam class (y=1): higher values across all features on average
X_spam = np.random.randn(n_samples // 2, 4) + np.array([2.0, 1.5, 1.0, 0.8])
# Ham class (y=0): centred at origin
X_ham  = np.random.randn(n_samples // 2, 4)

X_raw = np.vstack([X_spam, X_ham])
y_raw = np.hstack([np.ones(n_samples // 2), np.zeros(n_samples // 2)])

# Shuffle samples so the two classes are interleaved
shuffle_idx = np.random.permutation(n_samples)
X_raw, y_raw = X_raw[shuffle_idx], y_raw[shuffle_idx]

# Train / test split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Normalise features: fit scaler on training data only to avoid data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set: {X_train_sc.shape}  |  Test set: {X_test_sc.shape}')
print(f'Class balance (train) — spam: {y_train.mean():.1%}  ham: {1-y_train.mean():.1%}')

In [ ]:
# ── Train from-scratch model & plot convergence ───────────────────────────────
model_scratch = LogisticRegressionScratch(lr=0.1, n_iter=1000, lambda_=0.01)
model_scratch.fit(X_train_sc, y_train)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model_scratch.loss_history_, color='steelblue', linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Training Loss Curve — Logistic Regression from Scratch')
plt.tight_layout()
plt.show()

# ── Evaluate on test set ──────────────────────────────────────────────────────
y_pred_scratch = model_scratch.predict(X_test_sc)
print('\n--- From-Scratch Model Test Metrics ---')
print(f'Accuracy : {accuracy_score(y_test, y_pred_scratch):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_scratch):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_scratch):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_scratch):.4f}')

In [ ]:
# ── Compare against sklearn LogisticRegression ────────────────────────────────
# sklearn uses C = 1/lambda; set solver='lbfgs' for deterministic convergence
model_sklearn = LogisticRegression(
    C=1.0 / 0.01,          # equivalent L2 strength to our lambda_=0.01
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
model_sklearn.fit(X_train_sc, y_train)

# ── Coefficient comparison table ─────────────────────────────────────────────
feature_names = ['bias', 'word_freq_free', 'char_freq_!', 'capital_run_avg', 'link_count']
scratch_coef = model_scratch.theta_
sklearn_coef = np.concatenate([[model_sklearn.intercept_[0]], model_sklearn.coef_[0]])

coef_df = pd.DataFrame({
    'Feature'  : feature_names,
    'Scratch'  : np.round(scratch_coef, 4),
    'sklearn'  : np.round(sklearn_coef, 4),
    'Abs Diff' : np.round(np.abs(scratch_coef - sklearn_coef), 4)
})
print(coef_df.to_string(index=False))

# ── Closeness check (may not be exact due to solver differences) ──────────────
close = np.allclose(scratch_coef, sklearn_coef, atol=0.05)
print(f'\nnp.allclose(atol=0.05): {close}')
print(f'Max absolute difference: {np.max(np.abs(scratch_coef - sklearn_coef)):.4f}')
print('(Small differences expected: sklearn uses L-BFGS while we use gradient descent)')

In [ ]:
# ── Gradient verification via finite differences ──────────────────────────────
# If our analytical gradient formula is correct, it must match the numerical
# gradient estimated by perturbing each parameter by a tiny eps.

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def cross_entropy_loss(theta, X_b, y, lambda_=0.0):
    """Scalar cross-entropy loss at the given theta (with optional L2 penalty)."""
    y_hat = sigmoid(X_b @ theta)
    eps = 1e-15
    y_hat = np.clip(y_hat, eps, 1 - eps)
    data_loss = -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    # L2 penalty on all params except bias (index 0)
    reg_loss = (lambda_ / 2) * np.sum(theta[1:] ** 2)
    return data_loss + reg_loss

def numerical_gradient(theta, X_b, y, eps=1e-5, lambda_=0.0):
    """Estimate gradient by central finite differences."""
    grad = np.zeros_like(theta)
    for i in range(len(theta)):
        theta_plus  = theta.copy(); theta_plus[i]  += eps
        theta_minus = theta.copy(); theta_minus[i] -= eps
        grad[i] = (
            cross_entropy_loss(theta_plus,  X_b, y, lambda_) -
            cross_entropy_loss(theta_minus, X_b, y, lambda_)
        ) / (2 * eps)
    return grad

# Use a small random subset for speed
np.random.seed(0)
n_check = 50
idx     = np.random.choice(len(X_train_sc), n_check, replace=False)
X_chk   = X_train_sc[idx]
y_chk   = y_train[idx]
X_chk_b = np.column_stack([np.ones(n_check), X_chk])  # add bias column

# Evaluate at a known parameter vector (use trained weights for realism)
theta_chk = model_scratch.theta_.copy()
lambda_chk = 0.01

# Analytical gradient
y_hat_chk = sigmoid(X_chk_b @ theta_chk)
grad_analytical = (1 / n_check) * X_chk_b.T @ (y_hat_chk - y_chk)
# Add L2 regularisation gradient (skip bias)
reg_grad = lambda_chk * theta_chk.copy(); reg_grad[0] = 0
grad_analytical += reg_grad

# Numerical gradient
grad_numerical = numerical_gradient(theta_chk, X_chk_b, y_chk, lambda_=lambda_chk)

max_diff = np.max(np.abs(grad_analytical - grad_numerical))
print(f'Max absolute gradient difference: {max_diff:.2e}')
print(f'Gradient check passed (< 1e-5): {max_diff < 1e-5}')

---
## Section 2: Decision Boundary Visualization Gallery

We compare five classifiers on three synthetic datasets with very different geometries:

| Dataset | Geometry | Expected winner |
|---------|----------|-----------------|
| `make_blobs` | Linearly separable blobs | Logistic Regression |
| `make_moons` | Crescent / banana shapes | SVM-RBF or Random Forest |
| `make_circles` | Concentric rings | SVM-RBF or Random Forest |

In [ ]:
# ── Reusable decision-boundary plotting function ───────────────────────────────
def plot_decision_boundary(model, X, y, ax, title, proba=False, resolution=0.02):
    """
    Visualise a classifier's decision boundary on 2-D data.

    Parameters
    ----------
    model      : fitted sklearn-compatible classifier
    X          : 2-D feature array (n, 2)
    y          : integer label array (n,)
    ax         : matplotlib Axes to draw on
    title      : subplot title string
    proba      : if True, shade by predicted probability instead of class
    resolution : mesh grid step size (smaller = finer but slower)
    """
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    # Build a dense mesh grid covering the feature space
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, resolution),
        np.arange(y_min, y_max, resolution)
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]  # shape (M, 2)

    if proba and hasattr(model, 'predict_proba'):
        # Shade by P(y=1) to see soft probability landscape
        Z = model.predict_proba(grid_points)[:, 1]
        cmap_bg = 'RdBu_r'
    else:
        Z = model.predict(grid_points).astype(float)
        cmap_bg = 'coolwarm'

    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_bg)   # filled decision regions
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.2)  # decision boundary

    # Scatter training points, coloured by true class
    scatter_colors = ['#e06c75', '#61afef']            # red = class 0, blue = class 1
    for cls in np.unique(y):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=scatter_colors[int(cls)], s=18, edgecolors='white',
                   linewidths=0.4, label=f'Class {cls}', zorder=3)

    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Feature 1', fontsize=8)
    ax.set_ylabel('Feature 2', fontsize=8)
    ax.tick_params(labelsize=7)

print('plot_decision_boundary() defined.')

In [ ]:
# ── Generate the three synthetic 2-D datasets ─────────────────────────────────
np.random.seed(42)

X_blobs,   y_blobs   = make_blobs(n_samples=300, centers=2,      random_state=42, cluster_std=1.2)
X_moons,   y_moons   = make_moons(n_samples=300, noise=0.2,      random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1,    random_state=42, factor=0.5)

datasets = [
    ('Blobs (linear)',    X_blobs,   y_blobs),
    ('Moons (crescent)',  X_moons,   y_moons),
    ('Circles (rings)',   X_circles, y_circles),
]

# Normalise each dataset separately to aid convergence
datasets_sc = []
for name, X_d, y_d in datasets:
    sc = StandardScaler()
    datasets_sc.append((name, sc.fit_transform(X_d), y_d))

# ── Define the five classifiers ───────────────────────────────────────────────
classifiers = [
    ('Logistic Reg',    LogisticRegression(max_iter=1000, random_state=42)),
    ('k-NN (k=5)',      KNeighborsClassifier(n_neighbors=5)),
    ('Decision Tree',   DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('SVM RBF',         SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
    ('Random Forest',   RandomForestClassifier(n_estimators=100, random_state=42)),
]

print('Datasets and classifiers ready.')

In [ ]:
# ── 3×5 subplot grid: rows=datasets, cols=classifiers ────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
fig.suptitle('Decision Boundary Gallery: 5 Classifiers × 3 Datasets',
             fontsize=14, fontweight='bold', y=1.01)

acc_table = {}   # store accuracies for Section 7

for row_idx, (ds_name, X_d, y_d) in enumerate(datasets_sc):
    acc_table[ds_name] = {}
    for col_idx, (clf_name, clf) in enumerate(classifiers):
        clf.fit(X_d, y_d)                         # train on full dataset (visualisation)
        y_pred_d = clf.predict(X_d)
        acc = accuracy_score(y_d, y_pred_d)
        acc_table[ds_name][clf_name] = acc

        ax = axes[row_idx, col_idx]
        plot_decision_boundary(
            clf, X_d, y_d, ax,
            title=f'{clf_name}\n{ds_name}\nAcc={acc:.2%}'
        )

plt.tight_layout()
plt.show()

In [ ]:
# ── k-NN boundary evolution on make_moons: k = 1,3,5,15,50 ───────────────────
k_values = [1, 3, 5, 15, 50]
_, X_moons_sc, y_moons_sc = datasets_sc[1]   # pre-scaled moons dataset

fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
fig.suptitle('k-NN Decision Boundary Evolution on make_moons (k = 1 → 50)',
             fontsize=12, fontweight='bold')

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_moons_sc, y_moons_sc)
    train_acc = accuracy_score(y_moons_sc, knn.predict(X_moons_sc))
    plot_decision_boundary(
        knn, X_moons_sc, y_moons_sc, ax,
        title=f'k={k}  |  Train Acc={train_acc:.2%}'
    )

# Observation label
fig.text(0.5, -0.03,
         'k=1 → perfect training fit (high variance) | k=50 → smooth boundary (high bias)',
         ha='center', fontsize=10, style='italic')
plt.tight_layout()
plt.show()

In [ ]:
# ── Probability contour plots for Logistic Regression ────────────────────────
# Shows how confident the model is — not just which side of the boundary we're on
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Logistic Regression: P(y=1) Probability Landscape',
             fontsize=12, fontweight='bold')

for ax, (ds_name, X_d, y_d) in zip(axes, datasets_sc):
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_d, y_d)
    plot_decision_boundary(
        lr, X_d, y_d, ax,
        title=f'{ds_name}\n(probability shading)',
        proba=True              # shade by P(y=1) instead of hard class
    )

# Add a manual colourbar legend
sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label('P(y = 1)', fontsize=10)

plt.tight_layout()
plt.show()
print('Note: Logistic Regression can only learn a linear boundary.')
print('On make_circles and make_moons the probability landscape looks wrong.')

---
## Section 3: Numerical Gradient Verification

The analytical gradient of cross-entropy with L2 regularisation is:

$$\nabla_\theta \mathcal{L} = \frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{y}} - \mathbf{y}) + \lambda\boldsymbol{\theta}_{\text{no bias}}$$

We verify this against the **central finite-difference** approximation:
$$\frac{\partial \mathcal{L}}{\partial \theta_i} \approx \frac{\mathcal{L}(\theta_i + \varepsilon) - \mathcal{L}(\theta_i - \varepsilon)}{2\varepsilon}$$

In [ ]:
# ── Numerical gradient check (detailed output) ────────────────────────────────
# Reuse the helper functions defined in Section 1 (sigmoid, cross_entropy_loss,
# numerical_gradient).  Here we print a per-parameter comparison table.

np.random.seed(7)
n_v    = 80                                    # small subset for speed
idx_v  = np.random.choice(len(X_train_sc), n_v, replace=False)
Xv     = X_train_sc[idx_v]
yv     = y_train[idx_v]
Xv_b   = np.column_stack([np.ones(n_v), Xv])  # add bias column

# Use a random (not trained) theta to test the gradient at an arbitrary point
theta_v   = np.random.randn(Xv_b.shape[1]) * 0.5
lambda_v  = 0.05

# Analytical gradient
y_hat_v   = sigmoid(Xv_b @ theta_v)
g_anal    = (1 / n_v) * Xv_b.T @ (y_hat_v - yv)
reg_v     = lambda_v * theta_v.copy(); reg_v[0] = 0
g_anal   += reg_v

# Numerical gradient (central differences)
g_num = numerical_gradient(theta_v, Xv_b, yv, eps=1e-5, lambda_=lambda_v)

# Per-parameter comparison
param_names = ['bias'] + [f'theta_{i}' for i in range(1, Xv_b.shape[1])]
grad_df = pd.DataFrame({
    'Parameter' : param_names,
    'Analytical': np.round(g_anal, 6),
    'Numerical' : np.round(g_num,  6),
    'Abs Diff'  : np.round(np.abs(g_anal - g_num), 8)
})
print(grad_df.to_string(index=False))
print(f'\nMax abs diff: {np.max(np.abs(g_anal - g_num)):.2e}  (threshold: 1e-5)')
print(f'Gradient check PASSED: {np.max(np.abs(g_anal - g_num)) < 1e-5}')

In [ ]:
# ── Visualise gradient agreement ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: bar chart comparing analytical vs numerical gradient per parameter
x_pos = np.arange(len(g_anal))
width = 0.35
axes[0].bar(x_pos - width/2, g_anal, width, label='Analytical', color='steelblue', alpha=0.8)
axes[0].bar(x_pos + width/2, g_num,  width, label='Numerical',  color='coral',     alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(param_names, rotation=20)
axes[0].set_ylabel('Gradient value')
axes[0].set_title('Analytical vs Numerical Gradient')
axes[0].legend()

# Right: absolute difference per parameter (should all be near zero)
abs_diff = np.abs(g_anal - g_num)
axes[1].bar(x_pos, abs_diff, color='mediumseagreen', alpha=0.85)
axes[1].axhline(1e-5, color='red', linestyle='--', label='1e-5 threshold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(param_names, rotation=20)
axes[1].set_ylabel('|Analytical - Numerical|')
axes[1].set_title('Absolute Gradient Difference (should be < 1e-5)')
axes[1].legend()
axes[1].set_yscale('log')   # log scale to see small differences clearly

plt.tight_layout()
plt.show()

---
## Section 4: Multi-class Classification

We move beyond binary labels.  Two strategies for extending logistic regression to $K > 2$ classes:

| Strategy | How it works | Boundary |
|----------|-------------|----------|
| **One-vs-Rest (OvR)** | Train $K$ binary classifiers, each class vs all others | May leave ambiguous regions |
| **Softmax (Multinomial)** | Single model with $K$ output nodes; probabilities sum to 1 | No ambiguous regions |

In [ ]:
# ── 4-class blob dataset ──────────────────────────────────────────────────────
np.random.seed(42)
X_mc, y_mc = make_blobs(
    n_samples=400, centers=4, cluster_std=1.1, random_state=42
)

scaler_mc = StandardScaler()
X_mc_sc   = scaler_mc.fit_transform(X_mc)

# OvR: train 4 binary classifiers
clf_ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_ovr.fit(X_mc_sc, y_mc)

# Softmax (multinomial): single model with joint optimisation
clf_softmax = LogisticRegression(
    multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42
)
clf_softmax.fit(X_mc_sc, y_mc)

print(f'OvR Accuracy     : {accuracy_score(y_mc, clf_ovr.predict(X_mc_sc)):.4f}')
print(f'Softmax Accuracy : {accuracy_score(y_mc, clf_softmax.predict(X_mc_sc)):.4f}')

In [ ]:
# ── Multi-class decision boundary helper (4 colours) ─────────────────────────
def plot_mc_boundary(model, X, y, ax, title, resolution=0.03):
    """Decision boundary for multi-class (K>2) classifiers."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, resolution),
        np.arange(y_min, y_max, resolution)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.30, cmap='tab10', levels=np.arange(-0.5, 4.5, 1))
    ax.contour(xx,  yy, Z, levels=np.arange(-0.5, 4.5, 1), colors='k', linewidths=0.6)

    colors = ['#e06c75', '#61afef', '#98c379', '#e5c07b']
    for cls in np.unique(y):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[cls], s=20,
                   edgecolors='white', linewidths=0.4, label=f'Class {cls}', zorder=3)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Feature 1', fontsize=8)
    ax.set_ylabel('Feature 2', fontsize=8)
    ax.legend(fontsize=7, loc='best')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Multi-class Classification: OvR vs Softmax vs Confusion Matrix',
             fontsize=12, fontweight='bold')

plot_mc_boundary(clf_ovr,     X_mc_sc, y_mc, axes[0], 'One-vs-Rest (OvR)')
plot_mc_boundary(clf_softmax, X_mc_sc, y_mc, axes[1], 'Softmax (Multinomial)')

# Confusion matrix for softmax model
cm_mc = confusion_matrix(y_mc, clf_softmax.predict(X_mc_sc))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Pred {i}' for i in range(4)],
            yticklabels=[f'True {i}' for i in range(4)],
            ax=axes[2], cbar=False)
axes[2].set_title('Softmax Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# ── OvR ambiguous regions: where no classifier is confident (max prob < 0.5) ──
# In OvR, each binary classifier can output P < 0.5 for all classes simultaneously
# when the point lies between boundaries.

x_min, x_max = X_mc_sc[:, 0].min() - 0.5, X_mc_sc[:, 0].max() + 0.5
y_min, y_max = X_mc_sc[:, 1].min() - 0.5, X_mc_sc[:, 1].max() + 0.5
resolution   = 0.03

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, resolution),
    np.arange(y_min, y_max, resolution)
)
grid = np.c_[xx.ravel(), yy.ravel()]

# OvR: predicted probability of the winning class
proba_ovr = clf_ovr.predict_proba(grid)
max_proba  = proba_ovr.max(axis=1).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(xx, yy, max_proba, levels=20, cmap='YlOrRd_r', alpha=0.7)
ax.contour(xx, yy, max_proba, levels=[0.5], colors='black', linewidths=1.5,
           linestyles='--')
colors = ['#e06c75', '#61afef', '#98c379', '#e5c07b']
for cls in range(4):
    mask = y_mc == cls
    ax.scatter(X_mc_sc[mask, 0], X_mc_sc[mask, 1],
               c=colors[cls], s=20, edgecolors='white', linewidths=0.4, zorder=3)

plt.colorbar(cf, ax=ax, label='Max P(class) — OvR confidence')
ax.set_title('OvR Ambiguous Regions\n(dark red = low max probability = uncertain)',
             fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()
print('Dashed black contour = regions where the most confident OvR classifier'
      ' gives P < 0.5 (true ambiguity).')

---
## Section 5: MNIST Binary Classification — Digit 3 vs Digit 8

Digits **3** and **8** share similar strokes and pixel distributions, making them a challenging binary classification task (much harder than, say, 0 vs 1).  
We fetch the full MNIST dataset, filter to only these two classes, reduce to 2D with PCA for visualisation, and then compare Logistic Regression vs k-NN.

In [ ]:
# ── Load MNIST and filter to 3 vs 8 ──────────────────────────────────────────
print('Fetching MNIST from OpenML (may take ~30 s on first run) ...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist, y_mnist = mnist.data, mnist.target

# Keep only samples labelled '3' or '8'
mask_38  = (y_mnist == '3') | (y_mnist == '8')
X_38     = X_mnist[mask_38]                    # shape (~13 k, 784)
y_38     = (y_mnist[mask_38] == '8').astype(int)  # 1 = digit 8, 0 = digit 3

print(f'Filtered dataset: {X_38.shape[0]} samples')
print(f'Class balance — digit 3: {(y_38==0).sum()}  digit 8: {(y_38==1).sum()}')

# Pixel values already 0-255; normalise to [0, 1]
X_38 = X_38 / 255.0

# Train / test split
X_38_tr, X_38_te, y_38_tr, y_38_te = train_test_split(
    X_38, y_38, test_size=0.2, random_state=42, stratify=y_38
)
print(f'Train: {X_38_tr.shape}   Test: {X_38_te.shape}')

In [ ]:
# ── PCA: reduce 784 dims → 2 for visualisation ───────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
# Fit PCA on training data only, then project both sets
X_38_tr_2d = pca2.fit_transform(X_38_tr)
X_38_te_2d = pca2.transform(X_38_te)

print(f'Explained variance by first 2 PCs: {pca2.explained_variance_ratio_.sum():.1%}')

fig, ax = plt.subplots(figsize=(7, 5))
for label, color, marker in [(0, '#e06c75', 'o'), (1, '#61afef', '^')]:
    mask = y_38_tr == label
    digit = '3' if label == 0 else '8'
    ax.scatter(X_38_tr_2d[mask, 0], X_38_tr_2d[mask, 1],
               c=color, marker=marker, s=8, alpha=0.5, label=f'Digit {digit}')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('MNIST 3 vs 8 — PCA Projection to 2D')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Train LR and k-NN on raw 784-dim features ─────────────────────────────────
lr_mnist  = LogisticRegression(max_iter=500, solver='saga', random_state=42, C=0.1)
knn_mnist = KNeighborsClassifier(n_neighbors=5)

lr_mnist.fit(X_38_tr,  y_38_tr)
knn_mnist.fit(X_38_tr, y_38_tr)

y_lr_pred  = lr_mnist.predict(X_38_te)
y_knn_pred = knn_mnist.predict(X_38_te)

print('=== Logistic Regression (784 features) ===')
print(classification_report(y_38_te, y_lr_pred, target_names=['Digit 3', 'Digit 8']))

print('=== k-NN (k=5, 784 features) ===')
print(classification_report(y_38_te, y_knn_pred, target_names=['Digit 3', 'Digit 8']))

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_pred, name in zip(axes, [y_lr_pred, y_knn_pred],
                             ['Logistic Regression', 'k-NN (k=5)']):
    cm = confusion_matrix(y_38_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Pred 3', 'Pred 8'],
                yticklabels=['True 3', 'True 8'], ax=ax)
    ax.set_title(f'{name}\nAcc={accuracy_score(y_38_te, y_pred):.4f}')

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise misclassified digits from Logistic Regression ───────────────────
errors = np.where(y_lr_pred != y_38_te)[0]          # indices of test errors
show_n = min(10, len(errors))                         # show up to 10

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle(f'Logistic Regression — 10 Misclassified Digits (3 vs 8)',
             fontsize=12, fontweight='bold')

for ax, err_idx in zip(axes.ravel(), errors[:show_n]):
    img   = X_38_te[err_idx].reshape(28, 28)          # reshape flat vector to 28×28
    true_d  = '3' if y_38_te[err_idx] == 0 else '8'
    pred_d  = '3' if y_lr_pred[err_idx] == 0 else '8'
    ax.imshow(img, cmap='gray_r', interpolation='nearest')
    ax.set_title(f'True: {true_d}  Pred: {pred_d}', fontsize=9,
                 color='red' if true_d != pred_d else 'green')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Notice: most confusing samples are ambiguously written digits that')
print('even humans would struggle with at a glance.')

---
## Section 6: Threshold Tuning

The default threshold of **0.5** optimises accuracy, but this is rarely optimal in practice.  
For **spam detection with 5% spam prevalence** we care more about avoiding false negatives (missing spam) or false positives (blocking legitimate mail).  

Key metrics:
- **F1** = harmonic mean of precision and recall (equal weight)
- **F2** = $F_\beta$ with $\beta=2$ — recall is twice as important as precision

In [ ]:
# ── Class-imbalanced spam dataset: 5% spam ────────────────────────────────────
np.random.seed(42)
n_total = 2000
n_spam  = int(0.05 * n_total)   # 100 spam samples
n_ham   = n_total - n_spam      # 1900 ham samples

X_spam_imb = np.random.randn(n_spam,  4) + np.array([2.5, 2.0, 1.5, 1.2])
X_ham_imb  = np.random.randn(n_ham,   4)

X_imb = np.vstack([X_spam_imb, X_ham_imb])
y_imb = np.hstack([np.ones(n_spam), np.zeros(n_ham)])

shuffle = np.random.permutation(n_total)
X_imb, y_imb = X_imb[shuffle], y_imb[shuffle]

X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

scaler_imb  = StandardScaler()
X_imb_tr_sc = scaler_imb.fit_transform(X_imb_tr)
X_imb_te_sc = scaler_imb.transform(X_imb_te)

# class_weight='balanced' helps the model pay more attention to the minority class
lr_imb = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_imb.fit(X_imb_tr_sc, y_imb_tr)

y_scores = lr_imb.predict_proba(X_imb_te_sc)[:, 1]   # predicted P(spam)
print(f'Test set: {len(y_imb_te)} samples  |  spam: {int(y_imb_te.sum())} ({y_imb_te.mean():.1%})')

In [ ]:
# ── PR curve, ROC curve, and threshold sweep ───────────────────────────────────
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_imb_te, y_scores)
fpr_arr, tpr_arr, roc_thresholds         = roc_curve(y_imb_te, y_scores)
roc_auc_val = auc(fpr_arr, tpr_arr)
pr_auc_val  = auc(recall_arr, precision_arr)

# ── Threshold sweep ──────────────────────────────────────────────────────────
thresholds = np.linspace(0.05, 0.95, 100)
prec_sweep, rec_sweep, f1_sweep, f2_sweep = [], [], [], []

for t in thresholds:
    y_pred_t = (y_scores >= t).astype(int)
    prec_sweep.append(precision_score(y_imb_te, y_pred_t, zero_division=0))
    rec_sweep.append(recall_score(y_imb_te, y_pred_t, zero_division=0))
    f1_val = f1_score(y_imb_te, y_pred_t, zero_division=0)
    f1_sweep.append(f1_val)
    # F2 score: beta=2 (recall twice as important)
    p, r = prec_sweep[-1], rec_sweep[-1]
    f2_sweep.append((5 * p * r) / (4 * p + r) if (4 * p + r) > 0 else 0)

best_f1_thresh = thresholds[np.argmax(f1_sweep)]
best_f2_thresh = thresholds[np.argmax(f2_sweep)]

# ── 2×2 figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Threshold Analysis — Imbalanced Spam Dataset (5% spam)',
             fontsize=13, fontweight='bold')

# (0,0) Precision-Recall curve
ax = axes[0, 0]
ax.plot(recall_arr, precision_arr, color='purple', linewidth=2)
ax.fill_between(recall_arr, precision_arr, alpha=0.15, color='purple')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall Curve  (AUC-PR = {pr_auc_val:.3f})')
ax.axhline(y_imb_te.mean(), color='red', linestyle='--', label='Baseline (random)')
ax.legend()

# (0,1) ROC curve
ax = axes[0, 1]
ax.plot(fpr_arr, tpr_arr, color='steelblue', linewidth=2,
        label=f'AUC-ROC = {roc_auc_val:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
ax.fill_between(fpr_arr, tpr_arr, alpha=0.15, color='steelblue')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()

# (1,0) Precision, Recall, F1, F2 vs threshold
ax = axes[1, 0]
ax.plot(thresholds, prec_sweep, label='Precision', color='#e06c75')
ax.plot(thresholds, rec_sweep,  label='Recall',    color='#61afef')
ax.plot(thresholds, f1_sweep,   label='F1',        color='#98c379', linewidth=2)
ax.plot(thresholds, f2_sweep,   label='F2 (β=2)',  color='#e5c07b', linewidth=2, linestyle='--')
ax.axvline(best_f1_thresh, color='#98c379', linestyle=':', alpha=0.8,
           label=f'Best F1 thresh={best_f1_thresh:.2f}')
ax.axvline(best_f2_thresh, color='#e5c07b', linestyle=':', alpha=0.8,
           label=f'Best F2 thresh={best_f2_thresh:.2f}')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 / F2 vs Threshold')
ax.legend(fontsize=8)

# (1,1) Confusion matrices at three thresholds
ax = axes[1, 1]
ax.axis('off')
thresh_labels = [
    ('threshold=0.5',                           0.5),
    (f'Best F1 (t={best_f1_thresh:.2f})',       best_f1_thresh),
    (f'Best F2/recall (t={best_f2_thresh:.2f})',best_f2_thresh),
]
inner_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=axes[1, 1].get_subplotspec(), wspace=0.4)

for j, (lbl, thr) in enumerate(thresh_labels):
    inner_ax = fig.add_subplot(inner_gs[j])
    y_pred_t  = (y_scores >= thr).astype(int)
    cm_t      = confusion_matrix(y_imb_te, y_pred_t)
    sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
                ax=inner_ax)
    inner_ax.set_title(lbl, fontsize=7)
    inner_ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()
print(f'Best F1 threshold : {best_f1_thresh:.2f}  |  Best F2 threshold: {best_f2_thresh:.2f}')

In [ ]:
# ── Detailed metric comparison across three thresholds ────────────────────────
rows = []
for lbl, thr in [('Default (0.50)', 0.5),
                  (f'Best F1 ({best_f1_thresh:.2f})', best_f1_thresh),
                  (f'Best F2 ({best_f2_thresh:.2f})', best_f2_thresh)]:
    y_pred_t = (y_scores >= thr).astype(int)
    rows.append({
        'Threshold'  : lbl,
        'Accuracy'   : f'{accuracy_score(y_imb_te, y_pred_t):.4f}',
        'Precision'  : f'{precision_score(y_imb_te, y_pred_t, zero_division=0):.4f}',
        'Recall'     : f'{recall_score(y_imb_te, y_pred_t, zero_division=0):.4f}',
        'F1'         : f'{f1_score(y_imb_te, y_pred_t, zero_division=0):.4f}',
        'TP'         : int(confusion_matrix(y_imb_te, y_pred_t).ravel()[3]),
        'FP'         : int(confusion_matrix(y_imb_te, y_pred_t).ravel()[1]),
        'FN'         : int(confusion_matrix(y_imb_te, y_pred_t).ravel()[2]),
    })

print(pd.DataFrame(rows).to_string(index=False))
print('\nTakeaway: lowering the threshold catches more spam (higher recall) but')
print('at the cost of more legitimate emails being flagged (lower precision).')

---
## Section 7: Classifier Comparison Summary

We consolidate the accuracy numbers collected in Section 2 (full-dataset training) and interpret the results.

In [ ]:
# ── Accuracy table (datasets × classifiers) ───────────────────────────────────
clf_names = [name for name, _ in classifiers]
ds_names  = [name for name, _, _ in datasets_sc]

acc_matrix = pd.DataFrame(
    {clf: [acc_table[ds][clf] for ds in ds_names] for clf in clf_names},
    index=ds_names
)

print('Training Accuracy (%) — Full Dataset')
print((acc_matrix * 100).round(1).to_string())
print()

# Highlight the winner per dataset
print('Winner per dataset:')
for ds in ds_names:
    best_clf = acc_matrix.loc[ds].idxmax()
    best_acc = acc_matrix.loc[ds].max()
    print(f'  {ds:30s}  →  {best_clf}  ({best_acc:.1%})')

In [ ]:
# ── Grouped bar chart: accuracy per dataset per classifier ────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

n_clf = len(clf_names)
n_ds  = len(ds_names)
x     = np.arange(n_ds)
width = 0.15
colors_bar = ['#61afef', '#98c379', '#e5c07b', '#e06c75', '#c678dd']

for i, (clf_name, color) in enumerate(zip(clf_names, colors_bar)):
    vals = [acc_table[ds][clf_name] for ds in ds_names]
    offset = (i - n_clf / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=clf_name, color=color, alpha=0.85)
    # Annotate bars with accuracy value
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f'{h:.2f}',
                ha='center', va='bottom', fontsize=6.5)

ax.set_xticks(x)
ax.set_xticklabels(ds_names, fontsize=10)
ax.set_ylabel('Training Accuracy')
ax.set_ylim(0.7, 1.05)
ax.set_title('Classifier Accuracy Across Three Synthetic Datasets')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interpretation: why does each winner win? ─────────────────────────────────
print('=== Why does each classifier win? ===')
print()
print('Blobs (linearly separable):')
print('  Logistic Regression wins or ties because the data IS linearly separable.')
print('  Adding model complexity (Decision Tree, RF) does not help here.')
print()
print('Moons (crescent shapes):')
print('  Linear classifiers (LR) struggle because the boundary is curved.')
print('  k-NN, SVM-RBF, and RF all capture the non-linear boundary well.')
print('  k-NN with small k memorises every point; SVM-RBF uses the kernel trick.')
print()
print('Circles (concentric rings):')
print('  This is a radial geometry — no linear projection separates the classes.')
print('  SVM-RBF excels (the RBF kernel effectively adds x^2 features).')
print('  Random Forest also does well by building many axis-aligned splits.')
print('  Logistic Regression fails: it cannot find ANY good linear boundary.')
print()
print('Key takeaway: always match model capacity to dataset geometry.')
print('No single classifier wins on all datasets — the No Free Lunch theorem.')

---
## Section 8: Self-Check Questions

Test your understanding. Try to answer each question before expanding the hint.

---

**Q1.** Your from-scratch logistic regression gives different coefficients than sklearn after 1000 iterations. What are two possible causes?

<details>
<summary>Answer</summary>

1. **Solver difference**: Our implementation uses vanilla gradient descent; sklearn's default (`lbfgs`) uses a second-order quasi-Newton method that converges faster and to a higher-precision minimum. After 1000 iterations, gradient descent may not have converged fully, especially with a suboptimal learning rate.
2. **Regularisation mismatch**: sklearn uses `C = 1 / lambda` where the penalty is scaled differently (sometimes by `1/n`, sometimes not). Verify that `lambda_scratch == 1/C_sklearn` and that the normalisation convention (by `n` or `2n`) is the same on both sides.

Other valid causes: feature scaling differences, random initialisation if not zeroed, or numerical precision.
</details>

---

**Q2.** k-NN achieves 100% training accuracy on `make_moons` with k=1. What is its training error formula? Is it useful?

<details>
<summary>Answer</summary>

With **k=1**, each training point is its own nearest neighbour, so it always predicts its own label — training error = **0%** by definition.  
The formula: $\text{Training Error}_{k=1} = 0$ always (for any dataset without duplicate feature vectors with conflicting labels).

This is **not useful**: it tells us nothing about generalisation. The model has memorised the training data completely (variance = maximum, bias ≈ 0). On test data the boundary is jagged and noisy, leading to high variance and typically worse test accuracy than k=5 or k=15.
</details>

---

**Q3.** Logistic regression fails on `make_circles`. Instead of changing the model, you add features $x_1^2$ and $x_2^2$. Does this fix it? Why?

<details>
<summary>Answer</summary>

**Yes, it fixes it.** The concentric-circle boundary is $x_1^2 + x_2^2 = r^2$ (a circle), which is **not** linearly separable in $(x_1, x_2)$ space.  
After adding the polynomial features $\phi = [x_1, x_2, x_1^2, x_2^2]$, logistic regression now fits:

$$\theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1^2 + \theta_4 x_2^2 = 0$$

With $\theta_1 = \theta_2 = 0$ and $\theta_3 = \theta_4 > 0$ this becomes $x_1^2 + x_2^2 = c$, which is exactly the circle boundary. The model is still linear **in the lifted feature space** — this is the core idea behind the **kernel trick** used by SVM-RBF.
</details>

---

**Q4.** On MNIST 3-vs-8, your model confuses 3s and 8s. What visual features do these digits share that make this hard?

<details>
<summary>Answer</summary>

Digits 3 and 8 share:
- **Similar overall pixel density** in the same spatial regions (centre and right side of the image)
- **Curved strokes** at the top and bottom — a '3' is essentially an '8' with the left arcs removed
- **Similar bounding box** dimensions
- Handwriting variation: a sloppy 8 can have gaps that look like a 3; a 3 written with a closed loop resembles an 8

In PCA space, their clusters overlap significantly (much more than, say, 0 vs 1), so no simple boundary cleanly separates them. Convolutional Neural Networks learn stroke-level features that better capture the structural difference.
</details>

---

**Q5.** Your spam filter has AUC-PR = 0.45. Is this good? Compare to AUC-ROC = 0.95 on the same model. What does the discrepancy tell you?

<details>
<summary>Answer</summary>

**AUC-PR = 0.45 is poor.** The random baseline for AUC-PR equals the **prevalence** (fraction of positives). With 5% spam, the random baseline is 0.05. So 0.45 is better than random, but leaves a lot of room for improvement.

**AUC-ROC = 0.95 looks excellent** — but on imbalanced datasets it is misleadingly optimistic. ROC uses the true positive rate (sensitivity) and the false positive rate (1-specificity). With 95% negatives (ham), even a model that flags very few spam emails can achieve a low FPR while appearing to perform well.

**The discrepancy reveals class imbalance**: AUC-PR is much harder to inflate because precision is directly affected by the number of false positives relative to the (small) positive class. A model that rarely predicts positive will have high precision but very low recall — exposing poor AUC-PR. Always use AUC-PR (not AUC-ROC) as your primary metric for imbalanced classification tasks.
</details>